In [1]:
import glob
import math
import ROOT
import numpy as np
ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)

%jsroot

In [2]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/kitt/ALL_PHYS_THREE_PI0_NoSmearing/*.root"
)

all_files += glob.glob("/workspace/root_files/kitt/ALL_PHYS2_THREE_PI0_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/kitt/ALL_PHYS3_THREE_PI0_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/hal/DATA_THREE_PI0_NoSmearing/*.root")

for file in all_files:
    if '205' in file:
        all_files.remove(file)

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df = ROOT.RDataFrame("h1", all_files)
df_3pi0 = df.Filter("mcflag == 1 && (mctruth == 4 || mctruth == 0 || mctruth == -1)")
df_data = df.Define("Kchrec_5", "Kchrec[5]").Define("Knerec_5", "Knerec[5]").Define("KnerecSix_5", "KnerecSix[5]").Define("MassPi01", "pi01Fit[5]").Define("MassPi02", "pi02Fit[5]").Filter("mcflag == 0")
df = df.Define("Kchrec_5", "Kchrec[5]").Define("Knerec_5", "Knerec[5]").Define("KnerecSix_5", "KnerecSix[5]").Define("MassPi01", "pi0NeutralFit1[5]").Define("MassPi02", "pi0NeutralFit2[5]").Filter("mctruth != -1 && mctruth != 0 ")

In [3]:
counts = []

counts.append(df_3pi0.Count())
counts.append(df_3pi0.Filter("mctruth == 4 || mctruth == 0").Count())
counts.append(df_3pi0.Filter("mctruth == 4").Count())

errors = [300, 301, 303, 307, 308]
counts_errors = []
combined_error = ""

for error in errors:
    combined_error += f" && errorcode != {error}" if len(combined_error) > 0 else f"errorcode != {error}"
    counts_errors.append(df_3pi0.Filter(combined_error).Count())

all_counts = counts + counts_errors

ROOT.RDF.RunGraphs(all_counts)

for i, val in enumerate(counts):
    counts[i] = val.GetValue()

error_eff = []
rel_error_eff = []

for i, val in enumerate(counts_errors):
    counts_errors[i] = val.GetValue()
    eff = counts_errors[i] / counts[0] if counts[0] > 0 else 1.0
    rel_eff = counts_errors[i] / counts_errors[i-1] if i > 0 else eff
    error_eff.append(eff)
    rel_error_eff.append(rel_eff)

efficiencies = [0, 0, 0]
efficiencies[0] = counts[1] / counts[0] if counts[0] > 0 else 1.0
efficiencies[1] = counts[2] / counts[1] if counts[1] > 0 else 1.0
efficiencies[2] = counts[2] / counts[0] if counts[0] > 0 else 1.0

print(f"Counts total: {counts[0]},\nCounts preselection: {counts[1]},\nCounts selection: {counts[2]}")
print(f"Efficiency total: {efficiencies[2]*100:.2f}%,\nEfficiency preselection: {efficiencies[0]*100:.2f}%,\nEfficiency selection: {efficiencies[1]*100:.2f}%")

for i, eff in enumerate(error_eff):
    print(f"Error {errors[i]}: {eff*100:.2f}% ({rel_error_eff[i]*100:.2f}%) ({counts_errors[i]})")

Counts total: 26512811,
Counts preselection: 15040308,
Counts selection: 15040308
Efficiency total: 56.73%,
Efficiency preselection: 56.73%,
Efficiency selection: 100.00%
Error 300: 90.53% (90.53%) (24000846)
Error 301: 75.23% (83.11%) (19946094)
Error 303: 74.32% (98.78%) (19703367)
Error 307: 72.44% (97.48%) (19206772)
Error 308: 56.73% (78.31%) (15040308)


In [4]:
histos = []

histos.append(df.Histo1D(("h_nclu", ";nclu;Counts", 201, 0, 12), "nclu"))
ROOT.RDF.RunGraphs(histos)

1

In [5]:
canva = ROOT.TCanvas("canva", "", 800, 800)

histos[0].Draw()

canva.Draw()

## Optimization of the cut for the control sample tag $K\to\pi^+\pi^-$

In [6]:
df_CS_tag = {}

df_CS_tag["PM"] = df.Filter("has000 && mctruth != 0 && mctruth != -1")
df_CS_tag["nonPM"] = df.Filter("!has000 && mctruth != 0 && mctruth != -1")

total_counts = {}

total_counts["PM"] = df_CS_tag["PM"].Count().GetValue()
total_counts["nonPM"] = df_CS_tag["nonPM"].Count().GetValue()


### Create the histograms

In [7]:
cut_cond = "true"
optimized_variable = "bestErrorSixGamma"
var_title = "#Delta_{6#gamma} [cm^{2}]"

In [8]:
basic_hist = [df_CS_tag["PM"].Histo1D((f"h_{optimized_variable}", f";{var_title};Counts", 501, 0, 5000), optimized_variable)]
basic_hist.append(df_CS_tag["nonPM"].Histo1D((f"h_{optimized_variable}", f";{var_title};Counts", 501, 0, 5000), optimized_variable))

In [9]:
ROOT.RDF.RunGraphs(basic_hist)

1

In [10]:
mean = basic_hist[0].GetMean()
max_bin = basic_hist[0].GetMaximumBin()
max_x = basic_hist[0].GetBinCenter(max_bin)
max_y = basic_hist[0].GetBinContent(max_bin)
rms = basic_hist[0].GetRMS()

print(f"Max bin in: {max_x}")
print(f"RMS: {rms}")

canva = ROOT.TCanvas("canva", "", 800, 800)

# total_func = ROOT.TF1(
#     "total_gaus",
#     "crystalball",
#     490,
#     505,
# )

# total_func.SetParameter(0, basic_hist[0].GetMaximum()*0.7)
# total_func.SetParameter(3, basic_hist[0].GetMaximum()*0.3)

# total_func.SetParameter(1, basic_hist[0].GetMean())

# total_func.SetParameter(2, basic_hist[0].GetRMS())
# total_func.SetParameter(4, basic_hist[0].GetRMS())

# total_func.SetParLimits(0, 0.0, basic_hist[0].GetMaximum()*1.0)
# total_func.SetParLimits(3, 0.0, basic_hist[0].GetMaximum()*1.0)

# total_func.SetParLimits(2, basic_hist[0].GetRMS()*0.2, basic_hist[0].GetRMS()*1.0)
# total_func.SetParLimits(4, basic_hist[0].GetRMS()*1.0, basic_hist[0].GetRMS()*3.0)

# total_func.SetParNames("Core amplitude", "Mean", "Core Std Dev", "Tails amplitude", "Tails Std Dev")

# total_func.SetLineColor(ROOT.kBlack)


basic_hist[0].SetLineWidth(3)
basic_hist[0].SetLineColor(ROOT.kRed)
# basic_hist[0].Fit(total_func, "S", "", 495, 500)
basic_hist[0].Draw("HIST")
# total_func.Draw("SAME")
basic_hist[1].SetLineWidth(3)
basic_hist[1].SetLineColor(ROOT.kBlue)
basic_hist[1].Draw("SAME")

lineleft = ROOT.TLine(495.611, 0.0, 495.611, basic_hist[0].GetMaximum())
lineleft.SetLineStyle(2)
lineright = ROOT.TLine(499.611, 0.0, 499.611, basic_hist[0].GetMaximum())
lineright.SetLineStyle(2)
lineleft.Draw("SAME")
lineright.Draw("SAME")

# 3. NA KOŃCU dodajemy legendę (histogramy są już narysowane, więc ROOT je "widzi")
legend = ROOT.TLegend(0.5, 0.2, 0.85, 0.45)
legend.AddEntry(basic_hist[0].GetPtr(), "Events with primary K#rightarrow3#pi^{0}", "l")
legend.AddEntry(basic_hist[1].GetPtr(), "Events without primary K#rightarrow3#pi^{0}", "l")
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.Draw()

canva.Draw()

Max bin in: 424.15169660678646
RMS: 758.4890385708525


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canva


### Optimization loop

In [29]:
cut_values = [rms * float(val) for val in np.arange(0, 5.0, 0.1)]

pm_counts = []
nonpm_counts = []

for val in cut_values:
    condition = f"{optimized_variable} < {val}"

    pm_counts.append(df_CS_tag["PM"].Filter(condition).Count())
    nonpm_counts.append(df_CS_tag["nonPM"].Filter(condition).Count())

In [30]:
all_counts = pm_counts + nonpm_counts

ROOT.RDF.RunGraphs(all_counts)

1

In [31]:
pm_counts = [val.GetValue() for val in pm_counts]
nonpm_counts = [val.GetValue() for val in nonpm_counts]

In [32]:
purity = []
significance = []
efficiency = {"PM": [], "nonPM": []}

for nonpm, pm in zip(nonpm_counts, pm_counts):
    purity.append(pm / (nonpm + pm) if (nonpm + pm) > 0 else 0.99)
    significance.append(pm / np.sqrt(nonpm + pm) if (nonpm + pm) > 0 else 0.0)
    efficiency["PM"].append(pm / total_counts["PM"]) 
    efficiency["nonPM"].append(nonpm / total_counts["nonPM"]) 

In [33]:
# canva.SetRightMargin(0.3)

x_arr = np.array(cut_values, dtype=float)
y_arr = np.array(purity, dtype=float)

effpm_arr = np.array(efficiency["PM"], dtype=float)
effnonpm_arr = np.array(efficiency["nonPM"], dtype=float)

graph = ROOT.TGraph(len(x_arr), x_arr, y_arr)
graph_pm = ROOT.TGraph(len(x_arr), x_arr, effpm_arr)
graph_nonpm = ROOT.TGraph(len(x_arr), x_arr, effnonpm_arr)
graph.SetMaximum(1.2)
graph.SetMinimum(0.0)

graph.GetXaxis().SetTitle("Cut value #Delta_{6#gamma} [cm^{2}]")
graph.GetYaxis().SetTitle("Value [-]")
graph.SetLineWidth(2)
graph.SetTitle("")
graph.Draw("AL")

graph_pm.SetLineWidth(2)
graph_pm.SetLineColor(ROOT.kRed)
graph_pm.Draw("LSAME")

graph_nonpm.SetLineWidth(2)
graph_nonpm.SetLineColor(ROOT.kBlue)
graph_nonpm.Draw("LSAME")

line = ROOT.TLine(2.0, 0.0, 2.0, 1.2)
line.SetLineStyle(2)
line.Draw("SAME")

legend = ROOT.TLegend(0.5, 0.2, 0.85, 0.45)
legend.AddEntry(graph, "Purity K#rightarrow3#pi^{0}", "l")
legend.AddEntry(graph_pm, "Efficiency K#rightarrow3#pi^{0}", "l")
legend.AddEntry(graph_nonpm, "Efficiency non K#rightarrow3#pi^{0}", "l")
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.Draw()
canva.Draw()

In [35]:
optimal_purity = graph.Eval(1000.0)
optimal_efficiency = graph_pm.Eval(1000.0)
nonpm_eff = graph_nonpm.Eval(1000.0)

print(f"Optimal purity: {optimal_purity*100:.2f}%")
print(f"Optimal efficiency: {optimal_efficiency*100:.2f}%")
print(f"NonPM efficiency: {nonpm_eff*100:.2f}%")

Optimal purity: 95.05%
Optimal efficiency: 61.79%
NonPM efficiency: 2.82%


In [36]:
pm_counts_arr = np.array(pm_counts, dtype=float)
nonpm_counts_arr = np.array(nonpm_counts, dtype=float)

graph_pm_counts = ROOT.TGraph(len(x_arr), x_arr, pm_counts_arr)
graph_nonpm_counts = ROOT.TGraph(len(x_arr), x_arr, nonpm_counts_arr)

In [37]:
optimal_counts_pm = graph_pm_counts.Eval(1000.0)
optimal_counts_nonpm = graph_nonpm_counts.Eval(1000.0)

print(f"Optimal counts pm: {optimal_counts_pm:.0f}")
print(f"Optimal counts non pm: {optimal_counts_nonpm:.0f}")

Optimal counts pm: 9941699
Optimal counts non pm: 517563


In [38]:
df_tag = df.Filter(f"has000 && bestErrorSixGamma < 1000")
df_tag_data = df_data.Filter(f"bestErrorSixGamma < 1000")

In [39]:
condition_neutral_signal = "mctruth == 4"

In [40]:
counts = []

counts.append(df_tag.Count())
counts.append(df_tag.Filter(condition_neutral_signal).Count())

counts.append(df_tag_data.Count())

ROOT.RDF.RunGraphs(counts)

1

In [41]:
counts = [val.GetValue() for val in counts]

In [43]:
print(counts[0], counts[1], f"{counts[1] / counts[0] * 100:.2f}%")
print(counts[2], int(counts[2] * 0.9428))

9947401 9378490 94.28%
4678788 4411161


## Application of cuts

In [65]:
df_tag_3pi0 = df_tag.Filter("mctruth == 4")
df_tag_not_3pi0 = df_tag.Filter("mctruth != 4")

basis_3pi0_counts = df_tag_3pi0.Count().GetValue()
basis_data_counts = int(counts[2])

In [45]:
import math

pi0Mass1_mean = 134.83240924168848
pi0Mass2_mean = 134.87134080668446

# # 1. Define u and v as new columns in the RDataFrame using string expressions
df_tag_3pi0 = df_tag_3pi0.Define(
    "u", f"((MassPi01 - {pi0Mass1_mean}) + (MassPi02 - {pi0Mass2_mean})) / sqrt(2)"
).Define(
    "v", f"((MassPi01 - {pi0Mass1_mean}) - (MassPi02 - {pi0Mass2_mean})) / sqrt(2)"
)

df_tag_data = df_tag_data.Define(
    "u", f"((MassPi01 - {pi0Mass1_mean}) + (MassPi02 - {pi0Mass2_mean})) / sqrt(2)"
).Define(
    "v", f"((MassPi01 - {pi0Mass1_mean}) - (MassPi02 - {pi0Mass2_mean})) / sqrt(2)"
)

In [46]:
ROOT.gInterpreter.Declare("""
bool ellipse_cut(double u, double v) {
    double u0 = -92.9524, v0 = 7.1644, su = 33.7633, sv = 42.0163, rho = 0.556;
    double n_sigma_cut = 2.5;
    
    double du = u - u0;
    double dv = v - v0;
    double inv_rho2 = 1.0 / (1.0 - rho * rho);
    double dist2 = inv_rho2 * ((du * du) / (su * su) + (dv * dv) / (sv * sv) - 2.0 * rho * (du * dv) / (su * sv));
    return std::sqrt(std::max(0.0, dist2)) > n_sigma_cut;
}
""")

False

input_line_293:2:6: error: redefinition of 'ellipse_cut'
bool ellipse_cut(double u, double v) {
     ^
input_line_232:2:6: note: previous definition is here
bool ellipse_cut(double u, double v) {
     ^


In [60]:
# 2. Add your cuts, including any that use u and v
cuts = [
    "abs(Kchrec_5 - 497.605) < 2.637",
    "Chi2SignalKinFit < 30.0"
]

cut_counts = []
cut_histos = []
cut_eff = []
rel_cut_eff = []

combined_cut = ""

for cut in cuts:
    combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"
    cut_counts.append(df_tag_3pi0.Filter(combined_cut).Count())

ROOT.RDF.RunGraphs(cut_counts)

for i, count in enumerate(cut_counts):
    cut_counts[i] = count.GetValue()
    eff = cut_counts[i] / basis_3pi0_counts if basis_3pi0_counts > 0 else 1.0
    rel_eff = cut_counts[i] / cut_counts[i-1] if i > 0 else eff
    cut_eff.append(eff)
    rel_cut_eff.append(rel_eff)

    print(f"Cut {cuts[i]}: Counts {cut_counts[i]}, Efficiency {cut_eff[i]*100:.3f}%, Relative efficiency {rel_cut_eff[i]*100:.3f}%")



Cut abs(Kchrec_5 - 497.605) < 2.637: Counts 7525202, Efficiency 80.239%, Relative efficiency 80.239%
Cut Chi2SignalKinFit < 30.0: Counts 643928, Efficiency 6.866%, Relative efficiency 8.557%


In [62]:
# 2. Add your cuts, including any that use u and v

cut_counts_data = []
cut_eff_data = []
rel_cut_eff_data = []

combined_cut = ""

for cut in cuts:
    combined_cut += f" && ({cut})" if len(combined_cut) > 0 else f"({cut})"
    cut_counts_data.append(df_tag_data.Filter(combined_cut).Count())

ROOT.RDF.RunGraphs(cut_counts_data)

for i, count in enumerate(cut_counts_data):
    cut_counts_data[i] = count.GetValue()
    eff = cut_counts_data[i] / basis_data_counts if basis_data_counts > 0 else 1.0
    rel_eff = cut_counts_data[i] / cut_counts_data[i-1] if i > 0 and cut_counts_data[i-1] > 0 else eff
    cut_eff_data.append(eff)
    rel_cut_eff_data.append(rel_eff)

    print(f"Cut {cuts[i]}: Counts {cut_counts_data[i]}, Efficiency {cut_eff_data[i]*100:.3f}%, Relative efficiency {rel_cut_eff_data[i]*100:.3f}%")

Cut abs(Kchrec_5 - 497.605) < 2.637: Counts 3304424, Efficiency 70.626%, Relative efficiency 70.626%
Cut Chi2SignalKinFit < 30.0: Counts 282009, Efficiency 6.027%, Relative efficiency 8.534%


In [64]:
print(f"Correction factor: {cut_eff_data[-1] / cut_eff[-1]:.5f}")

Correction factor: 0.87786
